# DKD Risk Prediction in Type 2 Diabetes
## 03A. Corrected Data Audit and Final Model Alignment

This notebook prepares the corrected All of Us development data for reviewer-response model refitting.

The corrected landmark cohorts are the authoritative analytic populations. All participants who satisfy the corrected eligibility and landmark risk-set definitions are retained.

To maximize comparability with the submitted analysis:

- participants shared with the manuscript cohort retain their reconstructed original train/calibration/test assignment;
- participants no longer eligible under the corrected landmark definition are removed;
- newly eligible corrected-only participants are assigned deterministically with seed 42, within outcome strata, to keep the final partitions as close as possible to the original 60%/20%/20% allocation;
- shared participants whose DKD label changes remain in their original partition but use the corrected outcome label.

The archived fitted XGBoost models are used only to define the submitted model-specific predictor sets and predictor order. Final reviewer-response predictors are the intersection of the archived model predictors with predictors retained after corrected-data feature curation.

No model fitting is performed in this notebook.


# 1. Setup


In [ ]:
import json
import pickle
import warnings

import gcsfs
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

migration_folder = "rw-migration-aou-rw-639fade3"
bucket = migration_folder

COHORT_DIR = f"{bucket}/aou_dkd/dkd_cohort_20260821"
REFERENCE_DIR = f"{bucket}/aou_dkd/dkd_phenotypes_v3"
RESULT_DIR = f"{bucket}/aou_dkd/dkd_model_results_20260822"

fs = gcsfs.GCSFileSystem()

ID_COL = "person_id"
Y_COL = "dkd_binary"
FINAL_DATE = "20260822"

SPLIT_SEED = 42

GLOBAL_MISSING_THRESHOLD = 0.98
LANDMARK_MISSING_THRESHOLD = 0.98
RARE_LAB_MIN_MEASURED = 0.05
EARLY_LANDMARKS = ("30d", "6m")

# Columns retained in the Notebook 02 modeling matrices for cohort/QC
# traceability, but never eligible as predictors.
NON_PREDICTOR_METADATA = {
    "t2d_first_evidence_date",
    "cutoff_date",
    "kidney_first_evidence_date",
    "ehr_end_date",
    "landmark_name",
    "landmark_days",
}

print("Cohort directory:", COHORT_DIR)
print("Reference directory:", REFERENCE_DIR)
print("Results directory:", RESULT_DIR)


## I/O helpers


In [ ]:
def read_gcs_pickle(path):
    with fs.open(path, "rb") as f:
        return pd.read_pickle(f)


def load_gcs_pickle(path):
    with fs.open(path, "rb") as f:
        return pickle.load(f)


def save_gcs_pickle(obj, path):
    with fs.open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved pickle: {path}")


def save_gcs_csv(df, path):
    with fs.open(path, "w") as f:
        df.to_csv(f, index=False)
    print(f"Saved CSV: {path}")


def normalize_person_id(df, label):
    out = df.copy()

    out[ID_COL] = pd.to_numeric(
        out[ID_COL],
        errors="coerce",
    ).astype("Int64")

    if out[ID_COL].isna().any():
        raise ValueError(
            f"{label}: missing or non-numeric person_id."
        )

    if out[ID_COL].duplicated().any():
        raise ValueError(
            f"{label}: duplicate person_id."
        )

    return out


# 2. Load Manuscript and Corrected Landmark Data


In [ ]:
CORRECTED_PATHS = {
    "30d": (
        f"{COHORT_DIR}/"
        "DKD_Risk_Modeling_Data_Landmark_30d_20260822.pkl"
    ),
    "6m": (
        f"{COHORT_DIR}/"
        "DKD_Risk_Modeling_Data_Landmark_6m_20260822.pkl"
    ),
    "1y": (
        f"{COHORT_DIR}/"
        "DKD_Risk_Modeling_Data_Landmark_1y_20260822.pkl"
    ),
    "2y": (
        f"{COHORT_DIR}/"
        "DKD_Risk_Modeling_Data_Landmark_2y_20260822.pkl"
    ),
}

REFERENCE_PATHS = {
    "30d": (
        f"{REFERENCE_DIR}/"
        "dkd_risk_data_compact_surveys_baseline_v3.pkl"
    ),
    "6m": (
        f"{REFERENCE_DIR}/"
        "dkd_risk_data_compact_surveys_6m_v3.pkl"
    ),
    "1y": (
        f"{REFERENCE_DIR}/"
        "dkd_risk_data_compact_surveys_1y_v3.pkl"
    ),
    "2y": (
        f"{REFERENCE_DIR}/"
        "dkd_risk_data_compact_surveys_2y_v3.pkl"
    ),
}

for path in list(CORRECTED_PATHS.values()) + list(REFERENCE_PATHS.values()):
    if not fs.exists(path):
        raise FileNotFoundError(path)

corrected_raw = {
    landmark: normalize_person_id(
        read_gcs_pickle(path),
        f"corrected {landmark}",
    )
    for landmark, path in CORRECTED_PATHS.items()
}

reference_raw = {
    landmark: normalize_person_id(
        read_gcs_pickle(path),
        f"reference {landmark}",
    )
    for landmark, path in REFERENCE_PATHS.items()
}

print("\nCorrected landmark matrices")
for landmark, df in corrected_raw.items():
    print(
        f"  {landmark:>3}: "
        f"N={len(df):,} | "
        f"cases={int(df[Y_COL].sum()):,} | "
        f"columns={df.shape[1]:,}"
    )

print("\nArchived manuscript matrices")
for landmark, df in reference_raw.items():
    print(
        f"  {landmark:>3}: "
        f"N={len(df):,} | "
        f"cases={int(df[Y_COL].sum()):,} | "
        f"columns={df.shape[1]:,}"
    )


# 3. Cohort Revision Audit


In [ ]:
def build_cohort_audit(reference_df, corrected_df, landmark):
    ref = reference_df[[ID_COL, Y_COL]].copy()
    cor = corrected_df[[ID_COL, Y_COL]].copy()

    ref[ID_COL] = ref[ID_COL].astype("int64")
    cor[ID_COL] = cor[ID_COL].astype("int64")

    merged = (
        ref.rename(
            columns={Y_COL: "dkd_binary_reference"}
        )
        .merge(
            cor.rename(
                columns={Y_COL: "dkd_binary_corrected"}
            ),
            on=ID_COL,
            how="outer",
            indicator=True,
            validate="one_to_one",
        )
    )

    merged["membership"] = merged["_merge"].map(
        {
            "both": "shared",
            "left_only": "reference_only",
            "right_only": "corrected_only",
        }
    )

    merged["label_changed"] = (
        merged["membership"].eq("shared")
        & (
            merged["dkd_binary_reference"]
            != merged["dkd_binary_corrected"]
        )
    )

    summary = {
        "landmark": landmark,
        "reference_n": len(ref),
        "reference_cases": int(ref[Y_COL].sum()),
        "corrected_n": len(cor),
        "corrected_cases": int(cor[Y_COL].sum()),
        "shared_n": int(merged["membership"].eq("shared").sum()),
        "reference_only_n": int(
            merged["membership"].eq("reference_only").sum()
        ),
        "corrected_only_n": int(
            merged["membership"].eq("corrected_only").sum()
        ),
        "label_changed_shared_n": int(
            merged["label_changed"].sum()
        ),
    }

    return summary, merged.drop(columns="_merge")


cohort_rows = []
cohort_details = {}

for landmark in ["30d", "6m", "1y", "2y"]:
    summary, detail = build_cohort_audit(
        reference_raw[landmark],
        corrected_raw[landmark],
        landmark,
    )

    cohort_rows.append(summary)
    cohort_details[landmark] = detail

    save_gcs_csv(
        detail,
        f"{RESULT_DIR}/"
        f"cohort_transition_{landmark}_{FINAL_DATE}.csv",
    )

cohort_audit_df = pd.DataFrame(cohort_rows)

save_gcs_csv(
    cohort_audit_df,
    f"{RESULT_DIR}/"
    f"cohort_revision_audit_{FINAL_DATE}.csv",
)

print("\nCOHORT REVISION AUDIT")
print("=" * 110)
print(cohort_audit_df.to_string(index=False))


# 4. Recover the Original Model-Specific Predictor Sets


In [ ]:
ARCHIVED_MODEL_FILENAMES = {'30d': {'M1': 'dkd_base_M1_xgb_raw_20260304_062304.pkl', 'M2': 'dkd_base_M2_xgb_raw_20260304_062418.pkl', 'M3': 'dkd_base_M3_xgb_raw_20260304_062604.pkl', 'M4a': 'dkd_base_M4a_xgb_raw_20260304_062803.pkl', 'M5': 'dkd_base_M5_xgb_raw_20260304_063138.pkl'}, '6m': {'M1': 'dkd_m6_M1_xgb_raw_20260302_170949.pkl', 'M2': 'dkd_m6_M2_xgb_raw_20260302_171238.pkl', 'M3': 'dkd_m6_M3_xgb_raw_20260302_171446.pkl', 'M4a': 'dkd_m6_M4_xgb_raw_20260302_171711.pkl', 'M5': 'dkd_m6_M5_xgb_raw_20260302_172034.pkl'}, '1y': {'M1': 'dkd_y1_M1_xgb_raw_20260307_125023.pkl', 'M2': 'dkd_y1_M2_xgb_raw_20260307_125132.pkl', 'M3': 'dkd_y1_M3_xgb_raw_20260307_125338.pkl', 'M4': 'dkd_y1_M4_xgb_raw_20260307_125547.pkl', 'M5': 'dkd_y1_M5_xgb_raw_20260307_130106.pkl'}, '2y': {'M1': 'dkd_y2_M1_xgb_raw_20260302_174447.pkl', 'M2': 'dkd_y2_M2_xgb_raw_20260302_174554.pkl', 'M3': 'dkd_y2_M3_xgb_raw_20260302_174726.pkl', 'M4': 'dkd_y2_M4_xgb_raw_20260302_174928.pkl', 'M5': 'dkd_y2_M5_xgb_raw_20260302_175354.pkl'}}


In [ ]:
EXPECTED_REFERENCE_FEATURE_COUNTS = {'30d': {'M1': 94, 'M2': 123, 'M3': 169, 'M4a': 280, 'M5': 352}, '6m': {'M1': 95, 'M2': 124, 'M3': 170, 'M4a': 281, 'M5': 353}, '1y': {'M1': 95, 'M2': 124, 'M3': 170, 'M4': 396, 'M5': 468}, '2y': {'M1': 96, 'M2': 125, 'M3': 171, 'M4': 398, 'M5': 474}}


In [ ]:
ARCHIVED_MODEL_DIR_CANDIDATES = [
    f"{bucket}/aou_dkd/dkd_models_v4",
    f"{REFERENCE_DIR}/aou_dkd/dkd_models_v4",
    f"{REFERENCE_DIR}/dkd_models_v4",
]


def resolve_archived_model_path(landmark, model_id):
    filename = ARCHIVED_MODEL_FILENAMES[
        landmark
    ][
        model_id
    ]

    candidates = [
        f"{directory}/{filename}"
        for directory in ARCHIVED_MODEL_DIR_CANDIDATES
    ]

    for path in candidates:
        if fs.exists(path):
            return path

    raise FileNotFoundError(
        f"Could not locate archived {landmark}/{model_id} model.\n"
        + "\n".join(candidates)
    )


def extract_xgb_feature_names(model):
    if hasattr(model, "get_booster"):
        names = model.get_booster().feature_names
    elif hasattr(model, "feature_names_in_"):
        names = list(model.feature_names_in_)
    else:
        names = None

    if names is None:
        raise ValueError(
            "Archived model does not contain fitted feature names."
        )

    names = list(names)

    if len(names) != len(set(names)):
        raise ValueError(
            "Archived model contains duplicate feature names."
        )

    return names


archived_model_features = {}

for landmark, models in ARCHIVED_MODEL_FILENAMES.items():
    archived_model_features[landmark] = {}

    for model_id in models:
        model = load_gcs_pickle(
            resolve_archived_model_path(
                landmark,
                model_id,
            )
        )

        features = extract_xgb_feature_names(
            model
        )

        expected_n = EXPECTED_REFERENCE_FEATURE_COUNTS[
            landmark
        ][
            model_id
        ]

        if len(features) != expected_n:
            raise ValueError(
                f"{landmark}/{model_id}: archived model has "
                f"{len(features)} predictors; expected {expected_n}."
            )

        archived_model_features[
            landmark
        ][
            model_id
        ] = features

        print(
            f"{landmark:>3} | {model_id:>3} | "
            f"predictors={len(features):>3}"
        )


# 5. Preserve the Manuscript Split and Minimally Rebalance the Corrected Cohort

For participants shared between the manuscript and corrected cohorts, the manuscript 60%/20%/20% partition is reconstructed using the archived manuscript matrix and the original seed 42.

The corrected cohort is then formed as follows:

1. Shared participants initially retain their original partition.
2. Reference-only participants are removed because they no longer satisfy corrected eligibility.
3. Corrected-only participants are added deterministically with seed 42.
4. The final split is minimally rebalanced, within corrected case/control strata, to match **exactly** the sample sizes and class counts produced by the original two-stage stratified 60%/20%/20% procedure on the corrected cohort.
5. Only the minimum number of shared participants required to satisfy those exact targets are moved.
6. Shared participants whose DKD label changed use the corrected outcome label.

This preserves the manuscript partition for essentially the entire corrected cohort while retaining the full corrected risk set and exact split proportions.


In [ ]:

def reconstruct_reference_split(
    reference_df,
    *,
    seed=SPLIT_SEED,
):
    """
    Reconstruct the manuscript split from the archived matrix
    using the original two-stage stratified procedure.
    """
    x = reference_df[
        [
            ID_COL,
            Y_COL,
        ]
    ].copy()

    x[ID_COL] = x[ID_COL].astype("int64")

    ids_all = x[ID_COL].to_numpy()
    y_all = x[Y_COL].astype(int).to_numpy()

    ids_train, ids_temp, y_train, y_temp = train_test_split(
        ids_all,
        y_all,
        stratify=y_all,
        test_size=0.40,
        random_state=seed,
    )

    ids_cal, ids_test, y_cal, y_test = train_test_split(
        ids_temp,
        y_temp,
        stratify=y_temp,
        test_size=0.50,
        random_state=seed,
    )

    split_lookup = {}

    for split_name, ids in [
        ("train", ids_train),
        ("cal", ids_cal),
        ("test", ids_test),
    ]:
        for pid in ids:
            split_lookup[int(pid)] = split_name

    reference_label = {
        int(pid): int(label)
        for pid, label in zip(
            ids_all,
            y_all,
        )
    }

    return {
        "split_lookup": split_lookup,
        "reference_label": reference_label,
        "all_ids": set(map(int, ids_all)),
    }


def corrected_split_targets(
    corrected_df,
    *,
    seed=SPLIT_SEED,
):
    """
    Return only the target split-by-outcome counts that the
    original two-stage stratified 60/20/20 procedure would
    produce on the full corrected cohort.

    The resulting participant assignments are NOT used; only
    the target counts are used for minimal rebalancing.
    """
    x = (
        corrected_df[
            [
                ID_COL,
                Y_COL,
            ]
        ]
        .copy()
        .sort_values(
            ID_COL
        )
        .reset_index(
            drop=True
        )
    )

    ids_all = x[ID_COL].astype(int).to_numpy()
    y_all = x[Y_COL].astype(int).to_numpy()

    ids_train, ids_temp, y_train, y_temp = train_test_split(
        ids_all,
        y_all,
        stratify=y_all,
        test_size=0.40,
        random_state=seed,
    )

    ids_cal, ids_test, y_cal, y_test = train_test_split(
        ids_temp,
        y_temp,
        stratify=y_temp,
        test_size=0.50,
        random_state=seed,
    )

    targets = {}

    for split_name, y in [
        ("train", y_train),
        ("cal", y_cal),
        ("test", y_test),
    ]:
        targets[split_name] = {
            0: int((y == 0).sum()),
            1: int((y == 1).sum()),
        }

    return targets


def count_by_split_and_outcome(
    corrected_df,
    assignment,
):
    labels = {
        int(pid): int(y)
        for pid, y in zip(
            corrected_df[ID_COL],
            corrected_df[Y_COL],
        )
    }

    counts = {
        split: {
            0: 0,
            1: 0,
        }
        for split in [
            "train",
            "cal",
            "test",
        ]
    }

    for pid, split in assignment.items():
        y = labels[int(pid)]
        counts[split][y] += 1

    return counts


def build_hybrid_exact_split(
    corrected_df,
    reference_split_lookup,
    shared_ids,
    corrected_only_ids,
    *,
    seed=SPLIT_SEED,
):
    """
    Preserve all shared assignments initially, add corrected-only
    participants, then minimally move shared participants so the
    final split exactly matches the corrected cohort's stratified
    60/20/20 target counts.
    """
    corrected_df = corrected_df[
        [
            ID_COL,
            Y_COL,
        ]
    ].copy()

    corrected_df[ID_COL] = corrected_df[ID_COL].astype(int)
    corrected_df[Y_COL] = corrected_df[Y_COL].astype(int)

    shared_ids = set(map(int, shared_ids))
    corrected_only_ids = set(map(int, corrected_only_ids))

    labels = {
        int(pid): int(y)
        for pid, y in zip(
            corrected_df[ID_COL],
            corrected_df[Y_COL],
        )
    }

    targets = corrected_split_targets(
        corrected_df,
        seed=seed,
    )

    # Start by preserving all shared assignments.
    assignment = {
        pid: reference_split_lookup[pid]
        for pid in shared_ids
    }

    rng = np.random.default_rng(seed)

    # --------------------------------------------------------
    # Add corrected-only participants.
    # Assign within outcome strata to the split with the largest
    # remaining target deficit.
    # --------------------------------------------------------
    for y_value in [0, 1]:
        new_ids = np.array(
            sorted(
                pid
                for pid in corrected_only_ids
                if labels[pid] == y_value
            ),
            dtype=int,
        )

        if len(new_ids):
            new_ids = rng.permutation(
                new_ids
            )

        for pid in new_ids:
            current = count_by_split_and_outcome(
                corrected_df,
                assignment,
            )

            deficits = {
                split: (
                    targets[split][y_value]
                    - current[split][y_value]
                )
                for split in [
                    "train",
                    "cal",
                    "test",
                ]
            }

            chosen = max(
                [
                    "train",
                    "cal",
                    "test",
                ],
                key=lambda split: (
                    deficits[split],
                    {
                        "train": 2,
                        "cal": 1,
                        "test": 0,
                    }[split],
                ),
            )

            assignment[int(pid)] = chosen

    # --------------------------------------------------------
    # Minimal rebalancing of shared participants.
    # Work separately within corrected outcome strata so the
    # final case/control counts exactly match the stratified
    # corrected-cohort targets.
    # --------------------------------------------------------
    moved_shared = []

    for y_value in [0, 1]:
        while True:
            current = count_by_split_and_outcome(
                corrected_df,
                assignment,
            )

            delta = {
                split: (
                    current[split][y_value]
                    - targets[split][y_value]
                )
                for split in [
                    "train",
                    "cal",
                    "test",
                ]
            }

            excess_splits = [
                split
                for split, value in delta.items()
                if value > 0
            ]

            deficit_splits = [
                split
                for split, value in delta.items()
                if value < 0
            ]

            if not excess_splits and not deficit_splits:
                break

            if not excess_splits or not deficit_splits:
                raise RuntimeError(
                    f"Could not balance outcome stratum {y_value}: {delta}"
                )

            source = max(
                excess_splits,
                key=lambda split: (
                    delta[split],
                    {
                        "train": 2,
                        "cal": 1,
                        "test": 0,
                    }[split],
                ),
            )

            destination = min(
                deficit_splits,
                key=lambda split: (
                    delta[split],
                    {
                        "train": 0,
                        "cal": 1,
                        "test": 2,
                    }[split],
                ),
            )

            n_move = min(
                delta[source],
                -delta[destination],
            )

            # Prefer moving shared participants whose corrected
            # label did NOT change, preserving the 10 corrected-label
            # participants in their original partition whenever possible.
            source_shared = [
                pid
                for pid in shared_ids
                if (
                    labels[pid] == y_value
                    and assignment[pid] == source
                )
            ]

            stable_candidates = [
                pid
                for pid in source_shared
                if labels[pid]
                == reference_splits_current[
                    "reference_label"
                ].get(
                    pid,
                    labels[pid],
                )
            ]

            changed_label_candidates = [
                pid
                for pid in source_shared
                if pid not in set(
                    stable_candidates
                )
            ]

            stable_candidates = np.array(
                sorted(stable_candidates),
                dtype=int,
            )

            changed_label_candidates = np.array(
                sorted(changed_label_candidates),
                dtype=int,
            )

            if len(stable_candidates):
                stable_candidates = rng.permutation(
                    stable_candidates
                )

            if len(changed_label_candidates):
                changed_label_candidates = rng.permutation(
                    changed_label_candidates
                )

            candidate_order = list(
                stable_candidates
            ) + list(
                changed_label_candidates
            )

            if len(candidate_order) < n_move:
                raise RuntimeError(
                    f"Not enough shared candidates to move "
                    f"{n_move} from {source} to {destination} "
                    f"for outcome={y_value}."
                )

            for pid in candidate_order[:n_move]:
                pid = int(pid)

                assignment[pid] = destination

                moved_shared.append(
                    {
                        ID_COL: pid,
                        Y_COL: y_value,
                        "from_split": source,
                        "to_split": destination,
                    }
                )

    final_counts = count_by_split_and_outcome(
        corrected_df,
        assignment,
    )

    if final_counts != targets:
        raise RuntimeError(
            f"Final counts do not equal targets.\n"
            f"final={final_counts}\n"
            f"target={targets}"
        )

    return assignment, targets, pd.DataFrame(
        moved_shared
    )


reference_splits = {
    landmark: reconstruct_reference_split(
        reference_raw[landmark]
    )
    for landmark in [
        "30d",
        "6m",
        "1y",
        "2y",
    ]
}

corrected_development = {
    landmark: (
        df.copy()
        .sort_values(
            ID_COL
        )
        .reset_index(
            drop=True
        )
    )
    for landmark, df in corrected_raw.items()
}

hybrid_splits = {}
split_audit_rows = []
moved_shared_tables = {}

for landmark in [
    "30d",
    "6m",
    "1y",
    "2y",
]:
    corrected_df = corrected_development[
        landmark
    ]

    reference_ids = reference_splits[
        landmark
    ][
        "all_ids"
    ]

    corrected_ids = set(
        corrected_df[
            ID_COL
        ].astype(int)
    )

    shared_ids = (
        reference_ids
        & corrected_ids
    )

    corrected_only_ids = (
        corrected_ids
        - reference_ids
    )

    reference_only_ids = (
        reference_ids
        - corrected_ids
    )

    reference_lookup = reference_splits[
        landmark
    ][
        "split_lookup"
    ]

    # Used inside build_hybrid_exact_split to preferentially keep
    # the 10 corrected-label participants in their original split.
    reference_splits_current = reference_splits[
        landmark
    ]

    final_lookup, targets, moved_shared = build_hybrid_exact_split(
        corrected_df,
        reference_lookup,
        shared_ids,
        corrected_only_ids,
        seed=SPLIT_SEED,
    )

    moved_shared_tables[
        landmark
    ] = moved_shared

    save_gcs_csv(
        moved_shared,
        f"{RESULT_DIR}/"
        f"hybrid_split_moved_shared_{landmark}_{FINAL_DATE}.csv",
    )

    hybrid_splits[
        landmark
    ] = {
        "split_lookup": final_lookup,
        "shared_ids": shared_ids,
        "corrected_only_ids": corrected_only_ids,
        "reference_only_ids": reference_only_ids,
        "targets": targets,
    }

    x = corrected_df[
        [
            ID_COL,
            Y_COL,
        ]
    ].copy()

    x[
        "split"
    ] = (
        x[
            ID_COL
        ]
        .astype(int)
        .map(
            final_lookup
        )
    )

    split_tab = (
        x.groupby(
            "split"
        )[Y_COL]
        .agg(
            N="size",
            cases="sum",
        )
        .reindex(
            [
                "train",
                "cal",
                "test",
            ]
        )
    )

    split_tab[
        "controls"
    ] = (
        split_tab[
            "N"
        ]
        - split_tab[
            "cases"
        ]
    )

    split_tab[
        "pct"
    ] = (
        100
        * split_tab[
            "N"
        ]
        / len(x)
    )

    split_tab[
        "case_pct"
    ] = (
        100
        * split_tab[
            "cases"
        ]
        / split_tab[
            "N"
        ]
    )

    # Count only shared participants whose original partition had
    # to change during minimal rebalancing.
    moved_shared_n = len(
        moved_shared
    )

    print(
        f"\n{landmark}"
    )

    print(
        split_tab
    )

    print(
        "Target by split/outcome:",
        targets,
    )

    print(
        f"Shared participants minimally rebalanced: "
        f"{moved_shared_n:,}"
    )

    split_audit_rows.append(
        {
            "landmark": landmark,
            "reference_n": len(reference_ids),
            "corrected_n": len(corrected_ids),
            "shared_n": len(shared_ids),
            "reference_only_removed_n": len(
                reference_only_ids
            ),
            "corrected_only_added_n": len(
                corrected_only_ids
            ),
            "shared_rebalanced_n": moved_shared_n,
            "shared_rebalanced_pct_of_corrected": (
                100
                * moved_shared_n
                / len(corrected_ids)
            ),
            "train_n": int(
                split_tab.loc[
                    "train",
                    "N",
                ]
            ),
            "cal_n": int(
                split_tab.loc[
                    "cal",
                    "N",
                ]
            ),
            "test_n": int(
                split_tab.loc[
                    "test",
                    "N",
                ]
            ),
            "train_cases": int(
                split_tab.loc[
                    "train",
                    "cases",
                ]
            ),
            "cal_cases": int(
                split_tab.loc[
                    "cal",
                    "cases",
                ]
            ),
            "test_cases": int(
                split_tab.loc[
                    "test",
                    "cases",
                ]
            ),
        }
    )


hybrid_split_audit_df = pd.DataFrame(
    split_audit_rows
)

save_gcs_csv(
    hybrid_split_audit_df,
    f"{RESULT_DIR}/"
    f"hybrid_exact_split_audit_{FINAL_DATE}.csv",
)

print("\nHYBRID EXACT SPLIT AUDIT")
print("=" * 150)

print(
    hybrid_split_audit_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 30d and 6m cohort/outcome/split identity
# ------------------------------------------------------------
early_30d = (
    corrected_development[
        "30d"
    ][
        [
            ID_COL,
            Y_COL,
        ]
    ]
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)

early_6m = (
    corrected_development[
        "6m"
    ][
        [
            ID_COL,
            Y_COL,
        ]
    ]
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)

if not early_30d.equals(
    early_6m
):
    raise ValueError(
        "30d and 6m corrected cohorts/outcomes are not identical."
    )

if (
    hybrid_splits[
        "30d"
    ][
        "split_lookup"
    ]
    != hybrid_splits[
        "6m"
    ][
        "split_lookup"
    ]
):
    raise ValueError(
        "30d and 6m final split assignments are not identical."
    )

print(
    "\n30d/6m participant, outcome, and final split identity: PASSED"
)


# 6. Corrected Feature Curation


Feature curation is performed on the corrected development population before model alignment.

The following are removed:

- cohort-construction and outcome-timing metadata;
- features ≥98% missing across all corrected landmarks;
- features ≥98% missing at the current landmark;
- laboratory families measured in <5% of the landmark cohort;
- item-level SDoH psychological questions when prespecified composites are available;
- early-landmark trajectory/variability features;
- constant predictors;
- redundant `_no` indicators when a paired `_yes` indicator is present.

No missing categorical level is recreated.


In [ ]:
VAR_SUFFIXES = (
    "_sd",
    "_cov",
    "_slope",
    "_slope_dir",
    "_recency_days",
    "_ever_abnormal",
    "_frac_abnormal",
)

SDoH_PSYCH_BLOCKS = {
    "loneliness": {
        "item_prefixes": ["lonely_"],
        "keep": [
            "loneliness_mean",
            "loneliness_high",
            "loneliness_skip_any",
        ],
    },
    "harassment": {
        "item_prefixes": ["harass_"],
        "keep": [
            "harassment_score_mean",
            "harassment_freq_high",
            "harassment_skip_any",
        ],
    },
    "mistrust": {
        "item_prefixes": ["mistrust_"],
        "keep": [
            "mistrust_mean",
            "mistrust_high",
            "mistrust_skip_any",
        ],
    },
    "stress": {
        "item_prefixes": ["stress_"],
        "keep": [
            "stress_mean",
            "stress_high",
            "stress_skip_any",
        ],
    },
    "neighborhood_disorder": {
        "item_prefixes": ["disorder_"],
        "keep": [
            "neighborhood_disorder_mean",
            "neighborhood_disorder_skip_any",
        ],
    },
}

PROTECT_COLS = {
    ID_COL,
    Y_COL,
    "has_basic_survey",
    "has_lifestyle_survey",
    "has_sdoh_survey",
}


def find_yes_no_pairs(columns):
    cols = set(columns)
    pairs = []

    for col in columns:
        if col.endswith("_yes"):
            no_col = col[:-4] + "_no"

            if no_col in cols:
                pairs.append(
                    (col, no_col)
                )

    return pairs


def infer_lab_prefixes(df):
    return sorted(
        {
            c[:-2]
            for c in df.columns
            if c.endswith("_n")
        }
    )


def clean_corrected_landmarks(dfs):
    working = {
        landmark: df.replace(
            [np.inf, -np.inf],
            np.nan,
        ).copy()
        for landmark, df in dfs.items()
    }

    audit_rows = []

    # --------------------------------------------------------
    # 0. Remove cohort/outcome metadata explicitly
    # --------------------------------------------------------
    for landmark, df in working.items():
        present = sorted(
            NON_PREDICTOR_METADATA
            & set(df.columns)
        )

        for col in present:
            audit_rows.append(
                {
                    "landmark": landmark,
                    "feature": col,
                    "reason": "cohort_metadata",
                }
            )

        working[landmark] = df.drop(
            columns=present,
            errors="ignore",
        )

    # --------------------------------------------------------
    # 1. Global ultra-sparsity
    # --------------------------------------------------------
    shared_cols = set.intersection(
        *[
            set(df.columns)
            for df in working.values()
        ]
    )

    globally_sparse = []

    for col in sorted(shared_cols):
        if col in PROTECT_COLS:
            continue

        if all(
            df[col].isna().mean()
            >= GLOBAL_MISSING_THRESHOLD
            for df in working.values()
        ):
            globally_sparse.append(col)

    for landmark, df in working.items():
        for col in globally_sparse:
            if col in df.columns:
                audit_rows.append(
                    {
                        "landmark": landmark,
                        "feature": col,
                        "reason": "global_ultra_sparse",
                    }
                )

        working[landmark] = df.drop(
            columns=globally_sparse,
            errors="ignore",
        )

    # --------------------------------------------------------
    # 2. Landmark-specific cleaning
    # --------------------------------------------------------
    cleaned = {}

    for landmark, df in working.items():
        x = df.copy()

        # SDoH items
        sdoh_drop = []

        for spec in SDoH_PSYCH_BLOCKS.values():
            keep = set(
                spec["keep"]
            )

            for col in x.columns:
                if (
                    any(
                        col.startswith(prefix)
                        for prefix in spec["item_prefixes"]
                    )
                    and col not in keep
                ):
                    sdoh_drop.append(col)

        sdoh_drop = sorted(
            set(sdoh_drop)
        )

        for col in sdoh_drop:
            audit_rows.append(
                {
                    "landmark": landmark,
                    "feature": col,
                    "reason": "sdoh_item_removed",
                }
            )

        x = x.drop(
            columns=sdoh_drop,
            errors="ignore",
        )

        # Early landmark trajectory features
        if landmark in EARLY_LANDMARKS:
            early_drop = [
                col
                for col in x.columns
                if col.endswith(
                    VAR_SUFFIXES
                )
                and col not in PROTECT_COLS
            ]

            for col in early_drop:
                audit_rows.append(
                    {
                        "landmark": landmark,
                        "feature": col,
                        "reason": "early_landmark_trajectory_removed",
                    }
                )

            x = x.drop(
                columns=early_drop,
                errors="ignore",
            )

        # Rare lab families
        rare_family_drop = []

        for prefix in infer_lab_prefixes(x):
            n_col = f"{prefix}_n"

            frac_measured = float(
                (
                    pd.to_numeric(
                        x[n_col],
                        errors="coerce",
                    )
                    .fillna(0)
                    > 0
                ).mean()
            )

            if frac_measured < RARE_LAB_MIN_MEASURED:
                rare_family_drop.extend(
                    [
                        col
                        for col in x.columns
                        if col.startswith(
                            prefix + "_"
                        )
                    ]
                )

        rare_family_drop = sorted(
            set(rare_family_drop)
            - PROTECT_COLS
        )

        for col in rare_family_drop:
            audit_rows.append(
                {
                    "landmark": landmark,
                    "feature": col,
                    "reason": "rare_lab_family",
                }
            )

        x = x.drop(
            columns=rare_family_drop,
            errors="ignore",
        )

        # Landmark-specific ultra-sparsity
        missingness = x.isna().mean()

        sparse_drop = [
            col
            for col in missingness.index
            if (
                missingness[col]
                >= LANDMARK_MISSING_THRESHOLD
                and col not in PROTECT_COLS
            )
        ]

        for col in sparse_drop:
            audit_rows.append(
                {
                    "landmark": landmark,
                    "feature": col,
                    "reason": "landmark_ultra_sparse",
                }
            )

        x = x.drop(
            columns=sparse_drop,
            errors="ignore",
        )

        # Constant predictors
        constant_drop = []

        for col in x.columns:
            if col in PROTECT_COLS:
                continue

            non_na = x[col].dropna()

            if (
                not non_na.empty
                and non_na.nunique(
                    dropna=True
                )
                <= 1
            ):
                constant_drop.append(col)

        for col in constant_drop:
            audit_rows.append(
                {
                    "landmark": landmark,
                    "feature": col,
                    "reason": "constant",
                }
            )

        x = x.drop(
            columns=constant_drop,
            errors="ignore",
        )

        # Redundant paired yes/no indicators
        no_drop = [
            no_col
            for _, no_col in find_yes_no_pairs(
                x.columns
            )
        ]

        for col in no_drop:
            audit_rows.append(
                {
                    "landmark": landmark,
                    "feature": col,
                    "reason": "redundant_no_indicator",
                }
            )

        x = x.drop(
            columns=no_drop,
            errors="ignore",
        )

        cleaned[landmark] = x

    audit = (
        pd.DataFrame(audit_rows)
        .drop_duplicates()
        .sort_values(
            [
                "landmark",
                "reason",
                "feature",
            ]
        )
        .reset_index(drop=True)
    )

    return cleaned, audit


corrected_clean, feature_curation_audit = clean_corrected_landmarks(
    corrected_development
)

save_gcs_csv(
    feature_curation_audit,
    f"{RESULT_DIR}/"
    f"corrected_feature_curation_audit_{FINAL_DATE}.csv",
)

print("\nCORRECTED FEATURE CURATION")
print("=" * 90)

for landmark, df in corrected_clean.items():
    print(
        f"{landmark:>3}: "
        f"N={len(df):,} | "
        f"cases={int(df[Y_COL].sum()):,} | "
        f"columns={df.shape[1]:,}"
    )

print("\nRemoved features by reason:")
print(
    feature_curation_audit
    .groupby(
        [
            "landmark",
            "reason",
        ]
    )
    .size()
    .rename("n_removed")
    .reset_index()
    .to_string(index=False)
)


# 7. Align Each Model to the Cleaned Corrected Data


The archived model determines which predictor **families and variables were part of that submitted model**. The corrected-data curation determines which of those predictors remain eligible.

For each landmark-model pair:

`final predictors = archived fitted-model predictors ∩ corrected clean predictors`

The original archived feature order is preserved after filtering.

Corrected-only predictors are reported for audit but are not introduced into the reviewer-response models.


In [ ]:
final_model_features = {}
model_comparison_rows = []
model_comparison_detail = []

for landmark, models in archived_model_features.items():
    final_model_features[landmark] = {}

    clean_cols = set(
        corrected_clean[
            landmark
        ].columns
    )

    for model_id, archived_features in models.items():
        final_features = [
            feature
            for feature in archived_features
            if feature in clean_cols
        ]

        archived_only_dropped = [
            feature
            for feature in archived_features
            if feature not in clean_cols
        ]

        final_model_features[
            landmark
        ][
            model_id
        ] = final_features

        model_comparison_rows.append(
            {
                "landmark": landmark,
                "model_id": model_id,
                "archived_n_features": len(
                    archived_features
                ),
                "final_n_features": len(
                    final_features
                ),
                "dropped_n": len(
                    archived_only_dropped
                ),
            }
        )

        for feature in archived_features:
            model_comparison_detail.append(
                {
                    "landmark": landmark,
                    "model_id": model_id,
                    "feature": feature,
                    "status": (
                        "retained"
                        if feature in clean_cols
                        else "dropped_after_corrected_curation"
                    ),
                }
            )

model_feature_comparison_df = pd.DataFrame(
    model_comparison_rows
)

model_feature_comparison_detail_df = pd.DataFrame(
    model_comparison_detail
)

save_gcs_csv(
    model_feature_comparison_df,
    f"{RESULT_DIR}/"
    f"archived_vs_final_model_feature_counts_{FINAL_DATE}.csv",
)

save_gcs_csv(
    model_feature_comparison_detail_df,
    f"{RESULT_DIR}/"
    f"archived_vs_final_model_feature_detail_{FINAL_DATE}.csv",
)

print("\nARCHIVED vs FINAL REVIEWER-RESPONSE FEATURES")
print("=" * 100)
print(
    model_feature_comparison_df.to_string(
        index=False
    )
)


## Dropped archived predictors and curation reason


In [ ]:
dropped = (
    model_feature_comparison_detail_df.loc[
        model_feature_comparison_detail_df[
            "status"
        ].eq(
            "dropped_after_corrected_curation"
        )
    ]
    .merge(
        feature_curation_audit[
            [
                "landmark",
                "feature",
                "reason",
            ]
        ].drop_duplicates(),
        on=[
            "landmark",
            "feature",
        ],
        how="left",
    )
)

save_gcs_csv(
    dropped,
    f"{RESULT_DIR}/"
    f"archived_features_dropped_with_reason_{FINAL_DATE}.csv",
)

print(
    dropped.to_string(
        index=False
    )
)


## Corrected clean predictors not introduced into the reviewer-response models


In [ ]:
unused_corrected_rows = []

for landmark, df in corrected_clean.items():
    clean_predictors = set(
        df.columns
    ) - {
        ID_COL,
        Y_COL,
    }

    archived_union = set().union(
        *[
            set(features)
            for features in archived_model_features[
                landmark
            ].values()
        ]
    )

    unused = sorted(
        clean_predictors
        - archived_union
    )

    for feature in unused:
        unused_corrected_rows.append(
            {
                "landmark": landmark,
                "feature": feature,
            }
        )

    print(
        f"{landmark:>3}: corrected-clean predictors not used "
        f"by archived models = {len(unused)}"
    )

    if unused:
        print("  ", unused)

unused_corrected_features_df = pd.DataFrame(
    unused_corrected_rows
)

save_gcs_csv(
    unused_corrected_features_df,
    f"{RESULT_DIR}/"
    f"corrected_clean_features_not_introduced_{FINAL_DATE}.csv",
)


# 8. Build Final Model Matrices and Verify the Layer Structure


In [ ]:
final_model_data = {}

for landmark, models in final_model_features.items():
    final_model_data[landmark] = {}

    for model_id, features in models.items():
        df = corrected_clean[
            landmark
        ].loc[
            :,
            [
                ID_COL,
                Y_COL,
            ]
            + features,
        ].copy()

        if list(
            df.columns[
                2:
            ]
        ) != features:
            raise ValueError(
                f"{landmark}/{model_id}: feature order mismatch."
            )

        final_model_data[
            landmark
        ][
            model_id
        ] = df


# Check progressive nesting.
MODEL_ORDER = {
    "30d": ["M1", "M2", "M3", "M4a", "M5"],
    "6m": ["M1", "M2", "M3", "M4a", "M5"],
    "1y": ["M1", "M2", "M3", "M4", "M5"],
    "2y": ["M1", "M2", "M3", "M4", "M5"],
}

for landmark, order in MODEL_ORDER.items():
    previous = set()

    for model_id in order:
        current = set(
            final_model_features[
                landmark
            ][
                model_id
            ]
        )

        if not previous.issubset(
            current
        ):
            raise ValueError(
                f"{landmark}: {model_id} is not nested relative "
                "to the preceding layer."
            )

        previous = current

    print(
        f"{landmark:>3}: progressive layer nesting passed."
    )


# Explicit landmark-specific layer-name contract.
EXPECTED_MODEL_IDS = {
    "30d": {
        "M1",
        "M2",
        "M3",
        "M4a",
        "M5",
    },
    "6m": {
        "M1",
        "M2",
        "M3",
        "M4a",
        "M5",
    },
    "1y": {
        "M1",
        "M2",
        "M3",
        "M4",
        "M5",
    },
    "2y": {
        "M1",
        "M2",
        "M3",
        "M4",
        "M5",
    },
}

for landmark, expected_ids in EXPECTED_MODEL_IDS.items():
    actual_ids = set(
        final_model_features[
            landmark
        ].keys()
    )

    if actual_ids != expected_ids:
        raise ValueError(
            f"{landmark}: model IDs {sorted(actual_ids)} "
            f"do not match expected {sorted(expected_ids)}."
        )

print(
    "M4a/M4 landmark layer-name contract passed."
)


# 9. Hard Leakage and Metadata Gate


In [ ]:
FORBIDDEN_MODEL_FEATURES = set(
    NON_PREDICTOR_METADATA
) | {
    "event_or_censor_date",
    "last_followup_date",
    "first_activity_date",
    "days_kidney_after_t2d",
    "days_kidney_minus_t2d",
    "incident_case_candidate",
    "eligible_control",
    "early_kidney_event",
    "landmark_case",
    "landmark_control",
    "observable_at_landmark",
    "incident_dkd_on_or_before_landmark",
}

for landmark, models in final_model_features.items():
    for model_id, features in models.items():
        forbidden = sorted(
            set(features)
            & FORBIDDEN_MODEL_FEATURES
        )

        if forbidden:
            raise ValueError(
                f"{landmark}/{model_id}: forbidden cohort/outcome "
                f"metadata in predictors: {forbidden}"
            )

print(
    "Hard metadata/leakage gate passed for every final model."
)


# 10. Save Final Feature and Split Manifests


In [ ]:
feature_rows = []

for landmark, models in final_model_features.items():
    for model_id, features in models.items():
        for order, feature in enumerate(features):
            feature_rows.append(
                {
                    "landmark": landmark,
                    "model_id": model_id,
                    "feature_order": order,
                    "feature": feature,
                }
            )

final_feature_manifest = pd.DataFrame(
    feature_rows
)

FINAL_FEATURE_MANIFEST_PATH = (
    f"{RESULT_DIR}/"
    f"final_model_feature_manifest_{FINAL_DATE}.csv"
)

save_gcs_csv(
    final_feature_manifest,
    FINAL_FEATURE_MANIFEST_PATH,
)



split_rows = []

for landmark in [
    "30d",
    "6m",
    "1y",
    "2y",
]:
    df = corrected_clean[
        landmark
    ][
        [
            ID_COL,
            Y_COL,
        ]
    ].copy()

    split_info = hybrid_splits[
        landmark
    ]

    df[
        "landmark"
    ] = landmark

    df[
        "split"
    ] = (
        df[
            ID_COL
        ]
        .astype(
            int
        )
        .map(
            split_info[
                "split_lookup"
            ]
        )
    )

    df[
        "cohort_order"
    ] = (
        df[
            ID_COL
        ]
        .astype(int)
        .rank(
            method="first"
        )
        .astype(int)
        - 1
    )

    if df[
        "split"
    ].isna().any():
        raise ValueError(
            f"{landmark}: corrected participant missing split assignment."
        )

    split_rows.append(
        df[
            [
                "landmark",
                ID_COL,
                Y_COL,
                "split",
                "cohort_order",
            ]
        ]
    )

final_split_manifest = pd.concat(
    split_rows,
    ignore_index=True,
)

FINAL_SPLIT_MANIFEST_PATH = (
    f"{RESULT_DIR}/"
    f"final_model_split_manifest_{FINAL_DATE}.csv"
)

save_gcs_csv(
    final_split_manifest,
    FINAL_SPLIT_MANIFEST_PATH,
)


# 11. Save Final Reviewer-Response Modeling Datasets


One dataset is saved per landmark using the union of the final model-specific predictors. Because the information layers are nested, this is equivalent to the final M5 predictor set at that landmark.


In [ ]:
FINAL_MODEL_DATA_PATHS = {}

for landmark, order in MODEL_ORDER.items():
    final_features = final_model_features[
        landmark
    ][
        order[-1]
    ]

    df = corrected_clean[
        landmark
    ].loc[
        :,
        [
            ID_COL,
            Y_COL,
        ]
        + final_features,
    ].copy()

    # Deterministic storage order only; split membership is fixed above.
    df = (
        df.sort_values(
            ID_COL
        )
        .reset_index(
            drop=True
        )
    )

    pickle_path = (
        f"{COHORT_DIR}/"
        f"DKD_Risk_FinalModel_Data_Landmark_"
        f"{landmark}_{FINAL_DATE}.pkl"
    )

    parquet_path = (
        f"{COHORT_DIR}/"
        f"DKD_Risk_FinalModel_Data_Landmark_"
        f"{landmark}_{FINAL_DATE}.parquet"
    )

    save_gcs_pickle(
        df,
        pickle_path,
    )

    with fs.open(
        parquet_path,
        "wb",
    ) as f:
        df.to_parquet(
            f,
            index=False,
        )

    FINAL_MODEL_DATA_PATHS[
        landmark
    ] = {
        "pickle": pickle_path,
        "parquet": parquet_path,
    }

    print(
        f"{landmark:>3}: "
        f"N={len(df):,} | "
        f"cases={int(df[Y_COL].sum()):,} | "
        f"predictors={len(final_features):,}"
    )


# 12. Final Handoff Validation


In [ ]:
# Dataset/split identity.
for landmark, df in corrected_clean.items():
    data_ids = set(
        df[
            ID_COL
        ].astype(int)
    )

    manifest_ids = set(
        final_split_manifest.loc[
            final_split_manifest[
                "landmark"
            ].eq(
                landmark
            ),
            ID_COL,
        ].astype(int)
    )

    if data_ids != manifest_ids:
        raise ValueError(
            f"{landmark}: dataset/split participant mismatch."
        )


# Manifest identity and forbidden-feature check.
for landmark, models in final_model_features.items():
    for model_id, expected in models.items():
        manifest_features = (
            final_feature_manifest.loc[
                final_feature_manifest[
                    "landmark"
                ].eq(
                    landmark
                )
                & final_feature_manifest[
                    "model_id"
                ].eq(
                    model_id
                )
            ]
            .sort_values(
                "feature_order"
            )[
                "feature"
            ]
            .tolist()
        )

        if manifest_features != expected:
            raise ValueError(
                f"{landmark}/{model_id}: feature manifest mismatch."
            )

        if set(expected) & FORBIDDEN_MODEL_FEATURES:
            raise ValueError(
                f"{landmark}/{model_id}: forbidden metadata in final manifest."
            )


print("=" * 90)
print("FINAL REVIEWER-RESPONSE DATA HANDOFF PASSED")
print("=" * 90)

print(
    "Final predictors preserve the archived model specification "
    "where supported by corrected feature curation."
)

print(
    "No corrected-only predictors were introduced and no absent "
    "features were artificially recreated."
)

print(
    "Cohort/outcome timing metadata are excluded from every model."
)

print(
    "All corrected eligible participants are retained and "
    "assigned by the corrected 60/20/20 stratified split."
)

print("\nFinal modeling datasets:")
for landmark, paths in FINAL_MODEL_DATA_PATHS.items():
    print(
        f"  {landmark}: {paths['pickle']}"
    )

print("\nFeature manifest:")
print(
    FINAL_FEATURE_MANIFEST_PATH
)

print("\nSplit manifest:")
print(
    FINAL_SPLIT_MANIFEST_PATH
)


In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# Compare reconstructed manuscript split vs final corrected split
# ============================================================

SPLITS = ["train", "cal", "test"]

comparison_rows = []
detail_tables = {}

for landmark in ["30d", "6m", "1y", "2y"]:

    # -------------------------
    # Old / manuscript split
    # -------------------------
    ref_df = reference_raw[landmark][
        ["person_id", "dkd_binary"]
    ].copy()

    old_split_info = reconstruct_reference_split(
        ref_df,
        seed=SPLIT_SEED,
    )

    old_lookup = old_split_info["split_lookup"]

    old = ref_df.copy()
    old["person_id"] = old["person_id"].astype(int)
    old["old_split"] = old["person_id"].map(old_lookup)
    old = old.rename(
        columns={"dkd_binary": "old_label"}
    )

    # -------------------------
    # New / corrected split
    # -------------------------
    new_df = corrected_raw[landmark][
        ["person_id", "dkd_binary"]
    ].copy()

    new_df["person_id"] = new_df["person_id"].astype(int)

    new_lookup = hybrid_splits[landmark]["split_lookup"]

    new_df["new_split"] = new_df["person_id"].map(
        new_lookup
    )

    new_df = new_df.rename(
        columns={"dkd_binary": "new_label"}
    )

    # -------------------------
    # Merge
    # -------------------------
    comp = old.merge(
        new_df,
        on="person_id",
        how="outer",
        indicator=True,
        validate="one_to_one",
    )

    comp["status"] = comp["_merge"].map(
        {
            "both": "shared",
            "left_only": "old_only_removed",
            "right_only": "new_only_added",
        }
    )

    comp["split_changed"] = (
        comp["status"].eq("shared")
        & comp["old_split"].ne(comp["new_split"])
    )

    comp["label_changed"] = (
        comp["status"].eq("shared")
        & comp["old_label"].ne(comp["new_label"])
    )

    detail_tables[landmark] = comp.copy()

    # -------------------------
    # Summary
    # -------------------------
    shared = comp["status"].eq("shared")

    comparison_rows.append(
        {
            "landmark": landmark,
            "old_n": int(comp["old_split"].notna().sum()),
            "new_n": int(comp["new_split"].notna().sum()),
            "shared_n": int(shared.sum()),
            "old_only_removed_n": int(
                comp["status"].eq(
                    "old_only_removed"
                ).sum()
            ),
            "new_only_added_n": int(
                comp["status"].eq(
                    "new_only_added"
                ).sum()
            ),
            "shared_same_split_n": int(
                (
                    shared
                    & ~comp["split_changed"]
                ).sum()
            ),
            "shared_moved_split_n": int(
                comp["split_changed"].sum()
            ),
            "shared_moved_pct": (
                100
                * comp["split_changed"].sum()
                / shared.sum()
            ),
            "shared_label_changed_n": int(
                comp["label_changed"].sum()
            ),
        }
    )

    # -------------------------
    # Print detailed comparison
    # -------------------------
    print("\n" + "=" * 100)
    print(f"{landmark} SPLIT COMPARISON")
    print("=" * 100)

    print("\nOverall:")
    print(
        f"Old N:            {comp['old_split'].notna().sum():,}\n"
        f"New N:            {comp['new_split'].notna().sum():,}\n"
        f"Shared:           {shared.sum():,}\n"
        f"Removed:          {(comp['status'] == 'old_only_removed').sum():,}\n"
        f"Added:            {(comp['status'] == 'new_only_added').sum():,}\n"
        f"Shared moved:     {comp['split_changed'].sum():,}\n"
        f"Shared unchanged: {(shared & ~comp['split_changed']).sum():,}\n"
        f"Label changed:    {comp['label_changed'].sum():,}"
    )

    print("\nOld split counts:")
    print(
        comp.loc[
            comp["old_split"].notna(),
            "old_split",
        ]
        .value_counts()
        .reindex(SPLITS)
    )

    print("\nNew split counts:")
    print(
        comp.loc[
            comp["new_split"].notna(),
            "new_split",
        ]
        .value_counts()
        .reindex(SPLITS)
    )

    print("\nShared-participant transition table:")
    transition = pd.crosstab(
        comp.loc[shared, "old_split"],
        comp.loc[shared, "new_split"],
        rownames=["Old split"],
        colnames=["New split"],
    ).reindex(
        index=SPLITS,
        columns=SPLITS,
        fill_value=0,
    )

    print(transition)

    # New participants
    added = comp.loc[
        comp["status"].eq("new_only_added"),
        [
            "person_id",
            "new_label",
            "new_split",
        ],
    ].sort_values(
        ["new_split", "new_label", "person_id"]
    )

    print("\nNewly added participants by split:")
    print(
        added["new_split"]
        .value_counts()
        .reindex(SPLITS, fill_value=0)
    )

    # Removed participants
    removed = comp.loc[
        comp["status"].eq("old_only_removed"),
        [
            "person_id",
            "old_label",
            "old_split",
        ],
    ].sort_values(
        ["old_split", "old_label", "person_id"]
    )

    print("\nRemoved participants by old split:")
    print(
        removed["old_split"]
        .value_counts()
        .reindex(SPLITS, fill_value=0)
    )

    # Shared participants who were moved
    moved = comp.loc[
        comp["split_changed"],
        [
            "person_id",
            "old_label",
            "new_label",
            "old_split",
            "new_split",
            "label_changed",
        ],
    ].sort_values(
        ["old_split", "new_split", "person_id"]
    )

    print("\nShared participants moved between splits:")
    if moved.empty:
        print("None")
    else:
        print(moved.to_string(index=False))


# ============================================================
# Compact cross-landmark summary
# ============================================================

split_comparison_summary = pd.DataFrame(
    comparison_rows
)

print("\n" + "=" * 120)
print("CROSS-LANDMARK SPLIT COMPARISON SUMMARY")
print("=" * 120)
print(
    split_comparison_summary.to_string(
        index=False
    )
)

# Table 1

In [ ]:
# ========================================================================
# AoU Table 1 — matches BioMe Table 1 output schema exactly
# Reads the corrected-cohort final-model pickles from Notebook 03A
# Patches sex_male with OMOP person.sex_at_birth (near-100% coverage)
# ========================================================================
import numpy as np
import pandas as pd
import pickle
import gcsfs
from scipy import stats

# --- Paths and constants (mirror Notebook 03A) ------------------------
bucket         = "rw-migration-aou-rw-639fade3"
COHORT_DIR     = f"{bucket}/aou_dkd/dkd_cohort_20260821"
CURATION_DIR   = f"{bucket}/aou_dkd/dkd_data_curation_20260821"
RESULT_DIR     = f"{bucket}/aou_dkd/dkd_model_results_20260822"
FINAL_DATE     = "20260822"
ID_COL, Y_COL  = "person_id", "dkd_binary"
OUT_DATE       = "20261116"
OUT_PATH       = f"{RESULT_DIR}/aou_table1_all_landmarks_formatted_{OUT_DATE}.csv"

DEMO_PATH = f"{CURATION_DIR}/participant_demographics.parquet"

fs = gcsfs.GCSFileSystem()

def read_gcs_pickle(path):
    with fs.open(path, "rb") as f:
        return pickle.load(f)

def read_gcs_parquet(path):
    with fs.open(path, "rb") as f:
        return pd.read_parquet(f)

def save_gcs_csv(df, path):
    with fs.open(path, "w") as f:
        df.to_csv(f, index=False)

# --- 1. Load corrected landmark cohorts ------------------------------
LANDMARK_PATHS = {
    lm: f"{COHORT_DIR}/DKD_Risk_FinalModel_Data_Landmark_{lm}_{FINAL_DATE}.pkl"
    for lm in ["30d", "6m", "1y", "2y"]
}
cohorts = {lm: read_gcs_pickle(p) for lm, p in LANDMARK_PATHS.items()}

print("AoU corrected cohorts loaded:")
for lm, df in cohorts.items():
    n_cases = int(df[Y_COL].sum())
    n_ctrl  = len(df) - n_cases
    print(f"  {lm:>3}: N={len(df):,} | controls={n_ctrl:,} | "
          f"cases={n_cases:,} | prevalence={n_cases/len(df):.1%}")

# --- 2. Patch sex_male with OMOP sex_at_birth ------------------------
print(f"\nLoading OMOP demographics from {DEMO_PATH}")
demo = read_gcs_parquet(DEMO_PATH)
print(f"  demographics rows: {len(demo):,}")
print(f"  sex_at_birth value counts:")
print(demo["sex_at_birth"].value_counts(dropna=False).head(10).to_string())

# Map to binary. Only explicit Male/Female get encoded; everything else -> NaN.
demo = demo[["person_id", "sex_at_birth"]].drop_duplicates("person_id").copy()
demo["person_id"] = pd.to_numeric(demo["person_id"], errors="coerce").astype("Int64")
demo["sex_male_omop"] = (
    demo["sex_at_birth"]
    .map({"Male": 1, "Female": 0})
    .astype("Int64")
)
n_omop_coverage = int(demo["sex_male_omop"].notna().sum())
print(f"  OMOP sex_male coverage: {n_omop_coverage:,} / {len(demo):,} "
      f"({n_omop_coverage/len(demo):.1%})")

# Replace sex_male in each landmark cohort with the OMOP-sourced version.
print("\nPatching sex_male in landmark cohorts:")
for lm in cohorts:
    df = cohorts[lm].copy()
    df["person_id"] = pd.to_numeric(df["person_id"], errors="coerce").astype("Int64")
    # Drop old survey-derived sex_male and its missing flag if present
    drop_cols = [c for c in ["sex_male", "sex_male_missing"] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    df = df.merge(
        demo[["person_id", "sex_male_omop"]].rename(columns={"sex_male_omop": "sex_male"}),
        on="person_id", how="left",
    )
    cohorts[lm] = df
    n_nonmiss = int(df["sex_male"].notna().sum())
    print(f"  {lm:>3}: sex_male non-missing = {n_nonmiss:,} / {len(df):,} "
          f"({n_nonmiss/len(df):.1%})")

# --- 3. Variable spec (names match BioMe long CSV) -------------------
VAR_SPEC = [
    # (column_name_in_AoU_frame, display_label, variable_type)
    ("age_at_t2d",              "Age at first documented T2D, years",  "gauss"),
    ("HbA1c_last",              "HbA1c, %",                            "gauss"),
    ("eGFR_last",               "eGFR, mL/min/1.73 m^2",               "gauss"),
    ("Screatinine_last",        "Creatinine, mg/dL",                   "gauss"),
    ("UACR_last",               "UACR, mg/g",                          "skewed"),
    ("UrAlb_last",              "Urine albumin",                       "skewed"),
    ("SBP_last",                "SBP, mmHg",                           "gauss"),
    ("DBP_last",                "DBP, mmHg",                           "gauss"),
    ("LDL_last",                "LDL, mg/dL",                          "gauss"),
    ("TG_last",                 "Triglycerides, mg/dL",                "skewed"),
    ("sex_male",                "Male sex, %",                         "binary"),
    ("hypertension",            "Hypertension, %",                     "binary"),
    ("coronary_artery_disease", "CAD, %",                              "binary"),
    ("heart_failure",           "Heart failure, %",                    "binary"),
    ("atrial_fibrillation",     "Atrial fibrillation, %",              "binary"),
    ("obesity",                 "Obesity, %",                          "binary"),
    ("acute_kidney_injury",     "AKI history, %",                      "binary"),
    ("htn_acei_ever",           "ACE inhibitor exposure, %",           "binary"),
    ("htn_arb_ever",            "ARB exposure, %",                     "binary"),
    ("sglt2i_ever",             "SGLT2 inhibitor exposure, %",         "binary"),
    ("glp1ra_ever",             "GLP-1RA exposure, %",                 "binary"),
    ("insulin_ever",            "Insulin exposure, %",                 "binary"),
    ("lipid_statins_ever",      "Statin exposure, %",                  "binary"),
]

# --- 4. Sanity check: which spec columns are in the 1y frame? --------
_sample = cohorts["1y"]
_missing = [c for c, _, _ in VAR_SPEC if c not in _sample.columns]
if _missing:
    print("\n[warn] columns in VAR_SPEC but not in 1y frame — will be skipped:")
    for c in _missing:
        print(f"       {c}")
        candidates = [col for col in _sample.columns
                      if c.lower().split('_')[0] in col.lower()]
        if candidates:
            print(f"       candidates: {candidates[:5]}")

# --- 5. Formatters ---------------------------------------------------
def _mean_sd(x):
    x = pd.Series(x).dropna()
    return (f"{x.mean():.2f} ± {x.std():.2f}", len(x)) if len(x) else ("—", 0)

def _median_iqr(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return "—", 0
    q1, med, q3 = np.percentile(x, [25, 50, 75])
    return f"{med:.2f} ({q1:.2f}–{q3:.2f})", len(x)

def _binary_pct(x):
    x = pd.Series(x).dropna()
    return (f"{100*x.mean():.1f}%", len(x)) if len(x) else ("—", 0)

def _p_gauss(a, b):
    a = pd.Series(a).dropna().to_numpy()
    b = pd.Series(b).dropna().to_numpy()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return stats.ttest_ind(a, b, equal_var=False).pvalue

def _p_skewed(a, b):
    a = pd.Series(a).dropna().to_numpy()
    b = pd.Series(b).dropna().to_numpy()
    if len(a) == 0 or len(b) == 0:
        return np.nan
    return stats.mannwhitneyu(a, b, alternative="two-sided").pvalue

def _p_binary(a, b):
    a = pd.Series(a).dropna().astype(int)
    b = pd.Series(b).dropna().astype(int)
    if len(a) == 0 or len(b) == 0:
        return np.nan
    tbl = np.array([[(a == 1).sum(), (a == 0).sum()],
                    [(b == 1).sum(), (b == 0).sum()]])
    _, p, _, _ = stats.chi2_contingency(tbl)
    return p

def _fmt_p(p):
    if pd.isna(p): return "—"
    if p < 0.001:  return "<0.001"
    return f"{p:.3f}"

# --- 6. Build Table 1 ------------------------------------------------
rows = []
for lm in ["30d", "6m", "1y", "2y"]:
    df = cohorts[lm]
    ctrl = df.loc[df[Y_COL] == 0]
    case = df.loc[df[Y_COL] == 1]
    n_ctrl, n_case = len(ctrl), len(case)

    for col, display, vtype in VAR_SPEC:
        if col not in df.columns:
            continue
        if vtype == "gauss":
            vc, nc = _mean_sd(ctrl[col])
            vx, nx = _mean_sd(case[col])
            p = _p_gauss(ctrl[col], case[col])
        elif vtype == "skewed":
            vc, nc = _median_iqr(ctrl[col])
            vx, nx = _median_iqr(case[col])
            p = _p_skewed(ctrl[col], case[col])
        elif vtype == "binary":
            vc, nc = _binary_pct(ctrl[col])
            vx, nx = _binary_pct(case[col])
            p = _p_binary(ctrl[col], case[col])
        else:
            continue
        rows.append({
            "landmark": lm,
            "Controls_N": n_ctrl,
            "Cases_N": n_case,
            "Characteristic": display,
            "Controls": vc,
            "Cases": vx,
            "p-value": _fmt_p(p),
            "control_n_nonmissing": nc,
            "case_n_nonmissing": nx,
        })

aou_table1 = pd.DataFrame(rows)

# --- 7. Save and display ---------------------------------------------
save_gcs_csv(aou_table1, OUT_PATH)
print(f"\nWrote {OUT_PATH}  ({len(aou_table1)} rows)")

# 1y rows for quick review
aou_table1.loc[aou_table1["landmark"].eq("1y")].reset_index(drop=True)

# 13. Handoff to Notebook 03B

Notebook 03B consumes only:

- `DKD_Risk_FinalModel_Data_Landmark_<landmark>_20260822.pkl`
- `final_model_feature_manifest_20260822.csv`
- `final_model_split_manifest_20260822.csv`

The split manifest preserves the reconstructed manuscript partition for all shared participants except the minimum number required to restore the exact corrected-cohort 60%/20%/20% stratified target. Corrected-only participants are assigned deterministically with seed 42.
